In [ ]:
import torch
from sorl.gat_sim import GAT, GATConfig
torch.set_float32_matmul_precision('high')  # Enable TF32 for ~2x speedup

# Note: we reset 'BOS_TOKEN_ID' to 15 for copy & paste experiment
gat_config = GATConfig(
    vocab_sizes=[50256, 6],  # Level 0: 128 tokens, Level 1: 8 abstract tokens
    n_layer=12,
    n_head=6,
    n_embd=768,
    device="cuda" if torch.cuda.is_available() else "cpu",
    flex_kernel_options={
            "BLOCK_M": 32, "BLOCK_N": 32,
            "BLOCK_M1": 32, "BLOCK_N1": 64, "BLOCK_M2": 64, "BLOCK_N2": 32
        }
)
    
model = GAT(gat_config)
model = model.to("cuda")
model = torch.compile(model)

In [6]:
# model

In [ ]:
# heuristic rollout
import os, glob, itertools
from pathlib import Path

# MPS specific data loader functional (single device ver.)
# --------------------------------------------
def _load_data_shard(file: Path):
    header = torch.from_file(str(file), False, 256, dtype=torch.int32) # header is 256 int32
    assert header[0] == 20240520, "magic number mismatch in the data .bin file"
    assert header[1] == 1, "unsupported version"
    num_tokens = int(header[2]) # number of tokens (claimed)
    with file.open("rb", buffering=0) as f:
        tokens = torch.empty(num_tokens, dtype=torch.uint16, pin_memory=False) # MPS requires pin_memory=False
        f.seek(256 * 4)
        nbytes = f.readinto(tokens.numpy()) # avoid bytes->array copy by @YouJiacheng
        assert nbytes == 2 * num_tokens, "number of tokens read does not match header"
    return tokens

def data_generator(filename_pattern: str, sequence_length: int, device: str): 

    filename_pattern = "data/fineweb10B/fineweb_train_*.bin"
    files = [Path(file) for file in sorted(glob.glob(filename_pattern))]
    file_iter = itertools.cycle(files)
    tokens, pos = _load_data_shard(next(file_iter)), 0
    while True: 
        # Concern 1. Doesn't this means end-of-file is never reached?
        if pos + sequence_length + 1 >= len(tokens): # not enough data left -> load a new file
            tokens, pos = _load_data_shard(next(file_iter)), 0

        idx = tokens[pos : pos + sequence_length + 1].unsqueeze(0).to(device=device, dtype=torch.int32, non_blocking=True)
        pos += sequence_length
        yield idx

# ------------------------------------------------

# Question 1. Should we separate inputs / targets? 
#             That's really asking whether we want to 'reflect' on inputs, or inputs + next token
#             from the generation perspective, we ought to reflect on inputs and predict next-tok

# Reflection 1. 
# - based on above thought, we ought to modify the 'forward' method to take 'inputs' & 'targets' separately
#   the ._forward_pass and recursion should be done only on 'inputs'


data = _load_data_shard(Path("data/fineweb10B/fineweb_train_000002.bin"))

train_loader = data_generator(filename_pattern="data/fineweb10B/fineweb_train_*.bin", sequence_length=256, device="cuda")
val_loader = data_generator(filename_pattern="data/fineweb10B/fineweb_val_000000.bin", sequence_length=256, device="cuda")


In [ ]:
# --- Benchmark Speed & Memory Cost --- 
from sorl.benchmark import run_benchmark_suite
import torch 


# Prepare data - TEST WITH SMALLER SEQUENCE FIRST
tokens = next(train_loader)

# Run benchmark
results = run_benchmark_suite(
    model, 
    tokens, 
    memory_span=1024, 
    num_runs=10
)


############################################################
# GAT Benchmark Suite
############################################################
Sequence Length: 16
Batch Size: 1
Memory Span: 1024
Model Config: 768d, 12 layers, 6 heads

[1/5] Benchmarking: Forward (no grad)


AttributeError: module 'torch.mps' has no attribute 'current_device'

In [2]:
# ---- Copy & Paste Data Loader ----
from data.copy_paste import CopyPasteDataLoader

loader = CopyPasteDataLoader(vocab_size=16, max_token=10, seq_len=1, device='cpu')
tokens, loss_mask = loader.get_batch(1)

In [4]:
tokens

tensor([[15,  1, 16,  1, 15,  8, 16,  8, 15,  7, 16,  7, 15,  0, 16,  0, 15,  5,
         16,  5, 15,  2, 16,  2, 15,  9, 16,  9, 15,  8, 16,  8, 15,  7, 16,  7,
         15,  0, 16,  0, 15,  3, 16,  3, 15,  0, 16,  0, 15,  8, 16,  8, 15,  7,
         16,  7, 15,  0, 16,  0, 15,  4, 16,  4]])

In [3]:
from sorl.neo_utils import sorl_search, compute_loss, sorl_rollout, recursion
from sorl.gat_act import BOS_TOKEN_ID, recursion, infer_level
import torch, time
from sorl.neo_utils import select_best_per_doc

# training loop (pseudo version)
# Observation 1. 
# - with temperature = 1.0, search advantage (validated) is high enough 
# Reflection 1. 
# - therefore, the real question is how can we improve on trajectory perplexity? 
# Observation 2. 
# - again, setting 'memory_span' to 1 still gives near-perfect result

# Now the question is where is the limit of such 'memory compression'? 

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
batch_size = 16
memory_span = 1

for step in range(20): 
    optimizer.zero_grad()

    tokens, loss_mask = loader.get_batch(batch_size)
    # tokens = next(train_loader)

    # --- mixture of SoRL selection & deep supervision (avg. loss per iteration) ---
    with torch.no_grad(): 
        search_tokens, search_ppt, search_adv = sorl_search(tokens, model, n=4, K=3, max_iterations=1, memory_span=memory_span, temperature=1.0, loss_mask=loss_mask)
    
    # --- compute loss ---
    traj_loss, abs_loss = compute_loss(search_tokens, model, memory_span=memory_span, loss_mask=loss_mask)
    loss = traj_loss + abs_loss
    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 10 == 0: 
        with torch.no_grad(): 
            val_tokens, _, val_adv = sorl_search(tokens, model, n=4, K=3, max_iterations=1, memory_span=memory_span, temperature=10.0, loss_mask=loss_mask)
            traj_loss, abs_loss = compute_loss(val_tokens, model, memory_span=memory_span, loss_mask=loss_mask)
        print(f"validation step {step} | traj_loss: {traj_loss.item():.2f} | abs_loss: {abs_loss.item():.2f} | search adv: {val_adv.item() * 100:.2f}%")

validation step 0 | traj_loss: 0.67 | abs_loss: 1.96 | search adv: 6.30%
validation step 10 | traj_loss: 1.30 | abs_loss: 0.52 | search adv: 8.83%


In [11]:
# which abstraction corresponds to which token? 
# - we can scale up the 'seq len' & 'vocab size' to test the limit of "memory compression" ratio

In [12]:
search_tokens

tensor([[15,  5, 17,  5, 17, 15,  0, 17,  0, 17, 15,  2, 18,  2, 18, 15,  8, 17,
          8, 17, 15,  9, 17,  9, 17, 15,  1, 17,  1, 17, 15,  7, 17,  7, 17, 15,
          8, 17,  8, 17, 15,  1, 17,  1, 17, 15,  8, 17,  8, 17, 15,  2, 18,  2,
         18, 15,  2, 18,  2, 18, 15,  8, 17,  8]])

In [9]:
search_adv

tensor([0.0000, 0.0832])

In [ ]:
# Question 1. 
# - there are two ways of inference-time trick with SoRL
# - (1). use recursion, this has causal, parallel property for inference, but requires extra compute in training (include more recursion count)
# - (2). use search, this needs to be done per-token (search on prefix, compute next-token-loss), but no extra compute in training
# - TRM / HRM adopts (1). I also prefer (1). because the search advantage (greedy sample) will be big enough that (2) does not have any gains
# - This simplifies the inference process, too. 

# - Another thought is to re-use sorl_search in validation loop, the ACT halting gadget is also using 


# validation loop (pseudo version)
tokens = next(val_loader)

# === n > 1 includes search | max_iteration > 1 includes recursion | n_continuous doesn't affect causality ===
with torch.no_grad(): 
    aug_tokens, oracle_ppt, oracle_adv = sorl_search(tokens, model, n=3, K=3, max_iterations=1, n_continuous=0, memory_span=1024, temperature=1.0)

traj_loss, abs_loss = compute_loss(aug_tokens, model, oracle_ppt)



# SoRL Rollout Pipeline

The complete SoRL rollout pipeline consists of:

1. **`sorl_rollout`**: Generate n rollouts (1 greedy + n-1 stochastic)
   - Inserts rhythmic placeholder tokens at stride K
   - Fills placeholders using search with different temperatures
   - Returns all n rollouts

2. **`compute_perplexity_per_document`**: Evaluate quality of each rollout
   - Computes perplexity per document for each rollout
   - Lower perplexity = better prediction quality
   - Returns matrix of shape (n_rollouts, n_documents)

3. **`select_best_rollouts`**: Stitch best predictions together
   - Selects best rollout per document (lowest perplexity)
   - Stitches selected segments into final sequence
   - Returns single optimized sequence

This implements the SoRL algorithm where multiple rollouts are sampled and the best segments are selected based on perplexity.
